# 11 — Cross-Sectional XGBoost Ranker Tuning

This notebook searches for a stable combination of **feature columns** and **XGBRanker hyperparameters** before the discretionary replay study.

The workflow deliberately stays exploratory:

- one broad `FeatureSetSpec` is calculated once;
- candidate model schemas slice the loaded feature frame in memory;
- expanding folds are built only inside the outer training period;
- trials are scored on the highest-ranked stocks using future cross-sectional percentile and market-relative return;
- one selected configuration is fitted on the complete outer train split and checked once on outer validation;
- the locked test is never read.

The reported returns are research-target returns, not executable strategy P&L. Keep the committed notebook generic and export material runs to HTML before restoring the base state.


In [ ]:
from datetime import date
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from sklearn.model_selection import ParameterSampler
from xgboost import XGBRanker

from swingtrader.core.paths import find_repo_root
from swingtrader.data import features
from swingtrader.data.bronze.queries import load_available_tickers
from swingtrader.data.db import resolve_database_engine
from swingtrader.modeling.datasets import (
    CROSS_SECTIONAL_RETURN_PRIMARY_TASK,
    CROSS_SECTIONAL_RETURN_TARGET_SET,
    TemporalDatasetSpec,
    UniverseSpec,
    build_temporal_dataset,
    to_tabular_dataset,
)
from swingtrader.modeling.experiments import (
    FixedTemporalSplitter,
    TemporalCrossValidationSpec,
    TemporalSplitSpec,
    build_expanding_temporal_folds,
)
from swingtrader.modeling.training import (
    deterministic_random_scores,
    evaluate_cross_sectional_scores,
    prepare_xgboost_ranking_data,
)

RANDOM_SEED = 23
TOP_K = 10
HORIZON = 5
TOP_QUANTILE_THRESHOLD = 0.80
N_PARAMETER_SAMPLES_PER_FEATURE_SET = 2
INCLUDE_SLOW_MARKET_STRUCTURE = False
PROVIDER = "yfinance"

DATA_START = date(2008, 1, 1)
TRAIN_START = date(2010, 1, 1)
TRAIN_END = date(2020, 12, 31)
VALIDATION_START = date(2021, 1, 1)
VALIDATION_END = date(2023, 12, 31)
TEST_START = date(2024, 1, 1)
TEST_END = date(2025, 12, 31)

CV_SPEC = TemporalCrossValidationSpec(
    n_folds=4,
    validation_sessions=252,
    minimum_train_sessions=1_260,
)

repo_root = find_repo_root()
database_url = f"sqlite+pysqlite:///{(repo_root / 'data' / 'swingtrader.sqlite').as_posix()}"
ENGINE = resolve_database_engine(database_url=database_url)


## Resolve the exploratory universe

The notebook applies the current configured Stockholm Large/Mid Cap universe retrospectively. Results are therefore conditional on today's configured universe and may contain survivorship bias; they are not a reconstruction of historical point-in-time membership.


In [ ]:
def configured_tickers(path: Path) -> tuple[str, ...]:
    config = yaml.safe_load(path.read_text(encoding="utf-8"))
    return tuple(item["ticker"] for item in config["symbols"])


universe_directory = repo_root / "src" / "swingtrader" / "configs" / "universes"
configured = tuple(
    dict.fromkeys(
        configured_tickers(universe_directory / "se_large_cap.yml")
        + configured_tickers(universe_directory / "se_mid_cap.yml")
    )
)
available = set(
    load_available_tickers(
        engine=ENGINE,
        provider=PROVIDER,
        start_date=DATA_START,
        end_date=TEST_END,
    )
)
TICKERS = tuple(ticker for ticker in configured if ticker in available)

len(TICKERS), TICKERS[:5]


## Declare one broad feature set

Every enabled family is calculated once. Later trials only select columns from the resulting frame. The slow path-dependent market-structure block is disabled in the generic base notebook; enable it when its one-time loading cost is acceptable.


In [ ]:
feature_blocks = (
    features.FeatureBlockSpec(
        name="returns",
        builder=features.add_return_features,
        parameters={"horizons": (1, 5, 10, 20)},
        output_columns=("return_1d", "return_5d", "return_10d", "return_20d"),
        required_columns=frozenset({"adjusted_close"}),
    ),
    features.FeatureBlockSpec(
        name="cross_sectional",
        builder=features.add_cross_sectional_features,
        parameters={
            "return_horizons": (1, 5, 10, 20),
            "market_return_horizon": 1,
            "minimum_cross_section_size": 2,
        },
        output_columns=(
            "return_1d_cross_sectional_percentile",
            "return_5d_cross_sectional_percentile",
            "return_10d_cross_sectional_percentile",
            "return_20d_cross_sectional_percentile",
            "market_breadth_positive_1d",
            "market_mean_return_1d",
            "market_median_return_1d",
        ),
        required_columns=frozenset({"adjusted_close"}),
    ),
    features.FeatureBlockSpec(
        name="trend",
        builder=features.add_trend_features,
        parameters={
            "ma_lengths": (10, 20, 50),
            "adx_length": 14,
            "rolling_fraction_lookback": 20,
            "vwap_length": 20,
            "vwap_bollinger_length": 20,
            "vwap_bollinger_num_std": 2.0,
        },
        output_columns=(
            "ema_fast_to_ema_mid",
            "ema_mid_to_ema_slow",
            "ema_mid_to_sma_mid",
            "close_to_ema_fast",
            "close_to_ema_mid",
            "close_to_ema_slow",
            "close_over_ema_fast_fraction",
            "close_over_ema_mid_fraction",
            "close_over_ema_slow_fraction",
            "adx",
            "plus_di",
            "minus_di",
            "vwap_distance",
            "vwap_distance_percent_b",
        ),
        required_columns=frozenset(
            {"high", "low", "close", "volume", "adjusted_close"}
        ),
        history_requirement=features.HistoryRequirement.EXPANDING,
    ),
    features.FeatureBlockSpec(
        name="momentum",
        builder=features.add_momentum_features,
        parameters={
            "ppo_lengths": (12, 26, 9),
            "ppo_percentile_min_history": 100,
            "rsi_length": 21,
            "rsi_bollinger_length": 20,
            "rsi_bollinger_num_std": 2.0,
            "stochastic_k_length": 14,
            "stochastic_k_smoothing": 3,
            "stochastic_d_length": 3,
            "mfi_length": 14,
            "mfi_bollinger_length": 20,
            "mfi_bollinger_num_std": 2.0,
            "squeeze_bb_length": 20,
            "squeeze_bb_mult": 2.0,
            "squeeze_kc_length": 20,
            "squeeze_kc_mult": 1.5,
            "squeeze_atr_length": 14,
        },
        output_columns=(
            "ppo",
            "ppo_signal",
            "ppo_histogram",
            "ppo_percentile",
            "rsi",
            "rsi_percent_b",
            "stochastic_k",
            "stochastic_d",
            "mfi",
            "mfi_percent_b",
            "squeeze_on",
            "squeeze_off",
            "squeeze_released",
            "squeeze_width_ratio",
            "squeeze_momentum_atr",
            "squeeze_momentum_atr_change",
            "squeeze_duration",
            "squeeze_release_duration",
        ),
        required_columns=frozenset(
            {"high", "low", "close", "adjusted_close", "volume"}
        ),
        history_requirement=features.HistoryRequirement.EXPANDING,
    ),
    features.FeatureBlockSpec(
        name="volatility",
        builder=features.add_volatility_features,
        parameters={
            "adr_length": 20,
            "atr_length": 14,
            "bollinger_length": 20,
            "bollinger_num_std": 2.0,
        },
        output_columns=(
            "adr_percent",
            "atr_percent",
            "bollinger_bandwidth",
            "bollinger_percent_b",
        ),
        required_columns=frozenset({"high", "low", "close", "adjusted_close"}),
        history_requirement=features.HistoryRequirement.EXPANDING,
    ),
    features.FeatureBlockSpec(
        name="price_action",
        builder=features.add_price_action_features,
        parameters={
            "atr_length": 14,
            "range_percentile_length": 20,
            "breakout_length": 20,
            "rolling_candle_lookback": 14,
        },
        output_columns=(
            "candle_signed_body_fraction",
            "candle_upper_wick_fraction",
            "candle_lower_wick_fraction",
            "candle_close_location",
            "candle_range_atr",
            "candle_gap_atr",
            "range_percentile_20",
            "candle_inside_bar",
            "candle_outside_bar",
            "candle_engulfing_strength",
            "candle_lower_rejection_strength",
            "candle_upper_rejection_strength",
            "candle_consecutive_inside_bars",
            "candle_direction_run",
            "candle_direction_run_return",
            "candle_direction_run_body_atr",
            "candle_close_to_prior_high_atr_20",
            "candle_close_to_prior_low_atr_20",
            "candle_breakout_high_strength_20",
            "candle_breakout_low_strength_20",
            "candle_failed_breakout_high_strength_20",
            "candle_failed_breakout_low_strength_20",
            "rolling_bullish_candle_fraction",
        ),
        required_columns=frozenset(
            {"open", "high", "low", "close", "adjusted_close"}
        ),
        history_requirement=features.HistoryRequirement.EXPANDING,
    ),
    features.FeatureBlockSpec(
        name="volume",
        builder=features.add_volume_features,
        parameters={"turnover_zscore_length": 252, "turnover_zscore_log": True},
        output_columns=("turnover_zscore",),
        required_columns=frozenset({"close", "volume"}),
    ),
)

if INCLUDE_SLOW_MARKET_STRUCTURE:
    feature_blocks += (
        features.FeatureBlockSpec(
            name="market_structure",
            builder=features.add_market_structure_features,
            parameters={
                "donchian_length": 20,
                "zigzag_deviation": 5.0,
                "zigzag_pivot_legs": 10,
                "zigzag_consistency_pivots": 4,
                "zigzag_dynamics_legs": 6,
                "zigzag_atr_length": 14,
            },
            output_columns=(
                "donchian_position",
                "zigzag_last_direction",
                "zigzag_last_swing_return",
                "zigzag_last_swing_bars",
                "zigzag_swing_return_per_bar",
                "zigzag_bars_since_pivot",
                "zigzag_retracement",
                "market_structure_high_change",
                "market_structure_low_change",
                "market_structure_high_rate",
                "market_structure_low_rate",
                "market_structure_high_consistency",
                "market_structure_low_consistency",
                "market_structure_leg_balance",
                "market_structure_efficiency",
                "market_structure_close_to_prior_high_atr",
                "market_structure_close_to_prior_low_atr",
                "market_structure_breakout_high_strength",
                "market_structure_breakout_low_strength",
                "market_structure_failed_breakout_high_strength",
                "market_structure_failed_breakout_low_strength",
            ),
            required_columns=frozenset({"high", "low", "close"}),
            history_requirement=features.HistoryRequirement.PATH_DEPENDENT,
        ),
    )

BROAD_FEATURE_SET = features.FeatureSetSpec(
    name="cross_sectional_ranker_tuning_candidates",
    version="1",
    blocks=feature_blocks,
)

len(BROAD_FEATURE_SET.feature_columns), BROAD_FEATURE_SET.feature_columns[:8]


## Build and split the dataset once

`DATA_START` begins before `TRAIN_START` to provide warm-up history without loading the complete database. Expanding and path-dependent features are conditional on this chosen start date, so keep it fixed when comparing runs. Dataset construction is the expensive step; all tuning trials reuse the resulting frames.


In [ ]:
universe = UniverseSpec(
    name="stockholm_large_mid_cap_ranker_tuning",
    version="1",
    provider=PROVIDER,
    tickers=TICKERS,
)
dataset_spec = TemporalDatasetSpec(
    feature_set=BROAD_FEATURE_SET,
    target_set=CROSS_SECTIONAL_RETURN_TARGET_SET,
    task=CROSS_SECTIONAL_RETURN_PRIMARY_TASK,
    universe=universe,
    data_start=DATA_START,
    data_end=TEST_END,
)
split_spec = TemporalSplitSpec(
    name="cross_sectional_ranker_tuning_holdout",
    version="1",
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    validation_start=VALIDATION_START,
    validation_end=VALIDATION_END,
    test_start=TEST_START,
    test_end=TEST_END,
)

started = perf_counter()
bundle = build_temporal_dataset(engine=ENGINE, spec=dataset_spec)
split_result = FixedTemporalSplitter(split_spec).assign(bundle)
load_seconds = perf_counter() - started

tabular = to_tabular_dataset(bundle)
train_positions = split_result.indices("train")
validation_positions = split_result.indices("validation")

{
    "load_seconds": round(load_seconds, 1),
    "features": bundle.features.shape[1],
    "bundle_rows": len(bundle.features),
    "train_rows": len(train_positions),
    "validation_rows": len(validation_positions),
}


In [ ]:
relative_return_column = f"market_relative_forward_return_{HORIZON}d"
percentile_column = f"forward_return_{HORIZON}d_cross_sectional_percentile"
relevance_column = f"forward_return_{HORIZON}d_relevance_grade"

relative_return = bundle.targets[relative_return_column]
percentile = bundle.targets[percentile_column]
relevance = bundle.targets[relevance_column]

folds = build_expanding_temporal_folds(bundle, split_result, spec=CV_SPEC)
pd.DataFrame(
    [
        {
            "fold": fold.number,
            "train_start": fold.train_start,
            "train_end": fold.train_end,
            "validation_start": fold.validation_start,
            "validation_end": fold.validation_end,
            "train_rows": len(fold.train_indices),
            "validation_rows": len(fold.validation_indices),
        }
        for fold in folds
    ]
)


## Define in-memory feature candidates

Feature families come from the single broad specification. Candidate schemas preserve the canonical loaded column order. Add or remove a small number of interpretable candidates here rather than rebuilding the dataset for every trial.


In [ ]:
family_columns = {
    block.name: tuple(block.output_columns)
    for block in BROAD_FEATURE_SET.blocks
}
loaded_columns = tuple(tabular.X.columns)


def columns_without(*family_names: str) -> tuple[str, ...]:
    excluded = {
        column
        for family_name in family_names
        for column in family_columns[family_name]
    }
    return tuple(column for column in loaded_columns if column not in excluded)


compact_columns = (
    "return_10d",
    "return_20d",
    "return_5d_cross_sectional_percentile",
    "return_10d_cross_sectional_percentile",
    "return_20d_cross_sectional_percentile",
    "market_breadth_positive_1d",
    "ema_fast_to_ema_mid",
    "ema_mid_to_ema_slow",
    "close_to_ema_fast",
    "close_to_ema_mid",
    "close_to_ema_slow",
    "ppo",
    "rsi",
    "rsi_percent_b",
    "atr_percent",
    "bollinger_percent_b",
    "candle_range_atr",
    "candle_close_to_prior_high_atr_20",
    "candle_breakout_high_strength_20",
    "rolling_bullish_candle_fraction",
)

FEATURE_CANDIDATES = {
    "compact": compact_columns,
    "all_loaded": loaded_columns,
    "without_cross_sectional": columns_without("cross_sectional"),
    "without_momentum": columns_without("momentum"),
    "without_volatility": columns_without("volatility"),
    "without_price_action": columns_without("price_action"),
}

if INCLUDE_SLOW_MARKET_STRUCTURE:
    FEATURE_CANDIDATES["without_market_structure"] = columns_without(
        "market_structure"
    )

candidate_summary = pd.DataFrame(
    {
        "feature_count": {
            name: len(columns)
            for name, columns in FEATURE_CANDIDATES.items()
        },
        "columns": {
            name: ", ".join(columns)
            for name, columns in FEATURE_CANDIDATES.items()
        },
    }
)
candidate_summary


## Define a small joint search

The generic base uses deterministic random sampling rather than an exhaustive grid. Every feature candidate is evaluated with the same reference parameters and the same sampled hyperparameter configurations. This keeps the feature/parameter interaction visible without giving some feature candidates more search opportunities than others. Increase `N_PARAMETER_SAMPLES_PER_FEATURE_SET` only after confirming runtime.

`lambdarank_pair_method="topk"` and `lambdarank_num_pair_per_sample=TOP_K` align pair construction with the shortlist-oriented study.


In [ ]:
FIXED_RANKER_PARAMETERS = {
    "objective": "rank:ndcg",
    "eval_metric": f"ndcg@{TOP_K}",
    "tree_method": "hist",
    "lambdarank_pair_method": "topk",
    "lambdarank_num_pair_per_sample": TOP_K,
    "n_jobs": -1,
    "random_state": RANDOM_SEED,
    "verbosity": 0,
}

REFERENCE_PARAMETERS = {
    "n_estimators": 250,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 5,
    "subsample": 0.85,
    "colsample_bytree": 0.80,
    "gamma": 0.0,
    "reg_alpha": 0.0,
    "reg_lambda": 5.0,
}

PARAMETER_SPACE = {
    "n_estimators": [150, 250, 400],
    "learning_rate": [0.025, 0.05, 0.08],
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 5, 10],
    "subsample": [0.70, 0.85, 1.0],
    "colsample_bytree": [0.60, 0.80, 1.0],
    "gamma": [0.0, 0.1, 0.5],
    "reg_alpha": [0.0, 0.1, 1.0],
    "reg_lambda": [1.0, 5.0, 10.0],
    "lambdarank_normalization": [True, False],
}

sampled_parameter_sets = list(
    ParameterSampler(
        PARAMETER_SPACE,
        n_iter=N_PARAMETER_SAMPLES_PER_FEATURE_SET,
        random_state=RANDOM_SEED,
    )
)
trial_configs = []
for feature_candidate in FEATURE_CANDIDATES:
    trial_configs.append(
        {"feature_candidate": feature_candidate, **REFERENCE_PARAMETERS}
    )
    trial_configs.extend(
        {"feature_candidate": feature_candidate, **parameters}
        for parameters in sampled_parameter_sets
    )

# Remove accidental duplicate configurations while preserving order.
unique_configs = []
seen = set()
for config in trial_configs:
    key = tuple(sorted(config.items()))
    if key not in seen:
        seen.add(key)
        unique_configs.append(dict(config))
trial_configs = unique_configs

len(trial_configs), pd.DataFrame(trial_configs).head()


## Top-tail evaluation

The tuning metrics deliberately emphasize the selected shortlist rather than the complete universe:

- mean future cross-sectional percentile among the daily top `k`;
- mean and median market-relative forward return among the daily top `k`;
- fraction of selected stocks that actually finish in the future top quintile;
- fraction of dates where the selected basket has positive market-relative return.

The numeric XGBRanker score itself is not calibrated and is not used as a fixed threshold.


In [ ]:
def evaluate_top_tail(
    scores: pd.Series,
    future_percentile: pd.Series,
    future_relative_return: pd.Series,
    *,
    top_k: int,
    random_seed: int,
) -> tuple[pd.Series, pd.DataFrame]:
    if not scores.index.equals(future_percentile.index):
        raise ValueError("Scores and future percentiles must share an index.")
    if not scores.index.equals(future_relative_return.index):
        raise ValueError("Scores and future returns must share an index.")

    frame = pd.DataFrame(
        {
            "score": scores.astype("float64"),
            "future_percentile": future_percentile.astype("float64"),
            "future_relative_return": future_relative_return.astype("float64"),
        }
    )
    frame["tiebreak"] = deterministic_random_scores(
        frame.index,
        seed=random_seed,
    )

    rows = []
    for (provider, trading_date), group in frame.groupby(
        level=["provider", "trading_date"],
        sort=False,
    ):
        selected = group.sort_values(
            ["score", "tiebreak"],
            ascending=False,
            kind="stable",
        ).head(min(top_k, len(group)))
        rows.append(
            {
                "provider": provider,
                "trading_date": trading_date,
                "selected_count": len(selected),
                "top_k_mean_percentile": selected["future_percentile"].mean(),
                "top_k_mean_relative_return": selected[
                    "future_relative_return"
                ].mean(),
                "top_k_top_quintile_fraction": selected[
                    "future_percentile"
                ].ge(TOP_QUANTILE_THRESHOLD).mean(),
            }
        )

    daily = pd.DataFrame(rows).set_index(["provider", "trading_date"]).sort_index()
    summary = pd.Series(
        {
            "date_count": float(len(daily)),
            "mean_top_k_percentile": daily["top_k_mean_percentile"].mean(),
            "mean_top_k_relative_return": daily[
                "top_k_mean_relative_return"
            ].mean(),
            "median_top_k_relative_return": daily[
                "top_k_mean_relative_return"
            ].median(),
            "mean_top_k_top_quintile_fraction": daily[
                "top_k_top_quintile_fraction"
            ].mean(),
            "positive_top_k_date_fraction": daily[
                "top_k_mean_relative_return"
            ].gt(0).mean(),
        },
        dtype="float64",
    )
    return summary, daily


## Run the expanding-fold search


In [ ]:
fold_rows = []
trial_rows = []

for trial_id, config in enumerate(trial_configs, start=1):
    feature_candidate = str(config["feature_candidate"])
    selected_columns = FEATURE_CANDIDATES[feature_candidate]
    model_parameters = {
        key: value
        for key, value in config.items()
        if key != "feature_candidate"
    }
    trial_started = perf_counter()
    trial_fold_rows = []

    for fold in folds:
        X_fold_train = tabular.X.iloc[fold.train_indices].loc[
            :, list(selected_columns)
        ]
        X_fold_validation = tabular.X.iloc[fold.validation_indices].loc[
            :, list(selected_columns)
        ]
        relevance_fold_train = relevance.iloc[fold.train_indices]
        relevance_fold_validation = relevance.iloc[fold.validation_indices]

        X_rank_train, y_rank_train, qid_train = prepare_xgboost_ranking_data(
            X_fold_train,
            relevance_fold_train,
        )
        X_rank_validation, y_rank_validation, _ = prepare_xgboost_ranking_data(
            X_fold_validation,
            relevance_fold_validation,
        )

        model = XGBRanker(
            **FIXED_RANKER_PARAMETERS,
            **model_parameters,
        )
        model.fit(
            X_rank_train,
            y_rank_train,
            qid=qid_train,
            verbose=False,
        )
        scores = pd.Series(
            model.predict(X_rank_validation),
            index=X_rank_validation.index,
            dtype="float64",
            name="score",
        )
        top_tail_summary, _ = evaluate_top_tail(
            scores,
            percentile.iloc[fold.validation_indices].reindex(scores.index),
            relative_return.iloc[fold.validation_indices].reindex(scores.index),
            top_k=TOP_K,
            random_seed=RANDOM_SEED,
        )
        row = {
            "trial_id": trial_id,
            "fold": fold.number,
            "feature_candidate": feature_candidate,
            "feature_count": len(selected_columns),
            "validation_start": fold.validation_start,
            "validation_end": fold.validation_end,
            **top_tail_summary.to_dict(),
        }
        fold_rows.append(row)
        trial_fold_rows.append(row)

    trial_frame = pd.DataFrame(trial_fold_rows)
    trial_rows.append(
        {
            "trial_id": trial_id,
            "feature_candidate": feature_candidate,
            "feature_count": len(selected_columns),
            "mean_top_k_percentile": trial_frame[
                "mean_top_k_percentile"
            ].mean(),
            "mean_top_k_relative_return": trial_frame[
                "mean_top_k_relative_return"
            ].mean(),
            "median_fold_top_k_relative_return": trial_frame[
                "mean_top_k_relative_return"
            ].median(),
            "std_fold_top_k_relative_return": trial_frame[
                "mean_top_k_relative_return"
            ].std(ddof=0),
            "worst_fold_top_k_relative_return": trial_frame[
                "mean_top_k_relative_return"
            ].min(),
            "mean_top_k_top_quintile_fraction": trial_frame[
                "mean_top_k_top_quintile_fraction"
            ].mean(),
            "mean_positive_top_k_date_fraction": trial_frame[
                "positive_top_k_date_fraction"
            ].mean(),
            "elapsed_seconds": perf_counter() - trial_started,
            **model_parameters,
        }
    )

fold_results = pd.DataFrame(fold_rows)
leaderboard = pd.DataFrame(trial_rows)
leaderboard["return_rank"] = leaderboard["mean_top_k_relative_return"].rank(
    ascending=False,
    method="min",
)
leaderboard["percentile_rank"] = leaderboard["mean_top_k_percentile"].rank(
    ascending=False,
    method="min",
)
leaderboard["worst_fold_rank"] = leaderboard[
    "worst_fold_top_k_relative_return"
].rank(ascending=False, method="min")
leaderboard["selection_rank"] = leaderboard[
    ["return_rank", "percentile_rank", "worst_fold_rank"]
].mean(axis=1)
leaderboard = leaderboard.sort_values(
    ["selection_rank", "mean_top_k_relative_return"],
    ascending=[True, False],
).reset_index(drop=True)

leaderboard.head(15)


### Compare with a deterministic random shortlist

This baseline is calculated on the same inner-fold validation rows. It gives direct context for the top-tail percentile and relative-return levels.


In [ ]:
random_rows = []
for fold in folds:
    fold_index = tabular.X.iloc[fold.validation_indices].index
    random_scores = deterministic_random_scores(fold_index, seed=RANDOM_SEED)
    summary, _ = evaluate_top_tail(
        random_scores,
        percentile.iloc[fold.validation_indices],
        relative_return.iloc[fold.validation_indices],
        top_k=TOP_K,
        random_seed=RANDOM_SEED,
    )
    random_rows.append({"fold": fold.number, **summary.to_dict()})

random_fold_results = pd.DataFrame(random_rows)
random_fold_results


## Inspect stability before selecting a trial

The combined `selection_rank` is only a scale-free convenience: it gives equal weight to mean top-`k` relative return, mean top-`k` percentile, and the worst fold's top-`k` relative return. It is not a production objective. Prefer configurations with sensible results in every fold rather than one exceptional mean.


In [ ]:
top_trial_ids = leaderboard.head(8)["trial_id"]
plot_frame = fold_results[fold_results["trial_id"].isin(top_trial_ids)].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for trial_id, group in plot_frame.groupby("trial_id"):
    label = f"{trial_id}: {group['feature_candidate'].iloc[0]}"
    axes[0].plot(group["fold"], group["mean_top_k_relative_return"], marker="o", label=label)
    axes[1].plot(group["fold"], group["mean_top_k_percentile"], marker="o", label=label)

axes[0].axhline(
    random_fold_results["mean_top_k_relative_return"].mean(),
    linestyle="--",
    label="random mean",
)
axes[1].axhline(
    random_fold_results["mean_top_k_percentile"].mean(),
    linestyle="--",
    label="random mean",
)
axes[0].set_title(f"Top-{TOP_K} market-relative return by fold")
axes[1].set_title(f"Top-{TOP_K} future percentile by fold")
for axis in axes:
    axis.set_xlabel("Fold")
    axis.legend(fontsize=8)
plt.tight_layout()


## Choose one trial and evaluate outer validation once

The default below selects the first leaderboard row. During a real study, inspect the fold table and set `CHOSEN_TRIAL_ID` explicitly before running the remaining cells. Do not repeatedly change the configuration after reading outer-validation results.


In [ ]:
CHOSEN_TRIAL_ID = int(leaderboard.iloc[0]["trial_id"])
chosen_row = leaderboard.set_index("trial_id").loc[CHOSEN_TRIAL_ID]
chosen_config = trial_configs[CHOSEN_TRIAL_ID - 1]
chosen_feature_candidate = str(chosen_config["feature_candidate"])
chosen_columns = FEATURE_CANDIDATES[chosen_feature_candidate]
chosen_parameters = {
    key: value
    for key, value in chosen_config.items()
    if key != "feature_candidate"
}

chosen_row


In [ ]:
X_outer_train = tabular.X.iloc[train_positions].loc[:, list(chosen_columns)]
X_outer_validation = tabular.X.iloc[validation_positions].loc[
    :, list(chosen_columns)
]

X_rank_train, y_rank_train, qid_train = prepare_xgboost_ranking_data(
    X_outer_train,
    relevance.iloc[train_positions],
)
X_rank_validation, y_rank_validation, _ = prepare_xgboost_ranking_data(
    X_outer_validation,
    relevance.iloc[validation_positions],
)

chosen_model = XGBRanker(
    **FIXED_RANKER_PARAMETERS,
    **chosen_parameters,
)
chosen_model.fit(
    X_rank_train,
    y_rank_train,
    qid=qid_train,
    verbose=False,
)
validation_scores = pd.Series(
    chosen_model.predict(X_rank_validation),
    index=X_rank_validation.index,
    dtype="float64",
    name="score",
)

ranking_summary, ranking_daily = evaluate_cross_sectional_scores(
    validation_scores,
    y_rank_validation,
    relative_return.iloc[validation_positions].reindex(validation_scores.index),
    top_k=TOP_K,
    random_seed=RANDOM_SEED,
)
top_tail_summary, top_tail_daily = evaluate_top_tail(
    validation_scores,
    percentile.iloc[validation_positions].reindex(validation_scores.index),
    relative_return.iloc[validation_positions].reindex(validation_scores.index),
    top_k=TOP_K,
    random_seed=RANDOM_SEED,
)

pd.concat(
    {
        "whole_ranking": ranking_summary,
        "top_tail": top_tail_summary,
    },
    axis=1,
)


### Outer-validation stability by year


In [ ]:
validation_by_year = top_tail_daily.assign(
    year=top_tail_daily.index.get_level_values("trading_date").year
).groupby("year").agg(
    date_count=("top_k_mean_relative_return", "size"),
    mean_top_k_percentile=("top_k_mean_percentile", "mean"),
    mean_top_k_relative_return=("top_k_mean_relative_return", "mean"),
    median_top_k_relative_return=("top_k_mean_relative_return", "median"),
    positive_top_k_date_fraction=(
        "top_k_mean_relative_return",
        lambda values: values.gt(0).mean(),
    ),
    mean_top_k_top_quintile_fraction=(
        "top_k_top_quintile_fraction",
        "mean",
    ),
)
validation_by_year


### Inspect the extreme score tail

Ranker scores are not probabilities and their absolute scale can move when features or parameters change. The table therefore converts scores to a within-date percentile before comparing outcomes.


In [ ]:
validation_outcomes = pd.DataFrame(
    {
        "score": validation_scores,
        "future_percentile": percentile.iloc[validation_positions].reindex(
            validation_scores.index
        ),
        "future_relative_return": relative_return.iloc[validation_positions].reindex(
            validation_scores.index
        ),
    }
)
validation_outcomes["daily_score_percentile"] = validation_outcomes.groupby(
    level=["provider", "trading_date"]
)["score"].rank(method="average", pct=True)
validation_outcomes["score_bucket"] = pd.cut(
    validation_outcomes["daily_score_percentile"],
    bins=[0.0, 0.50, 0.80, 0.90, 0.95, 0.98, 1.0],
    include_lowest=True,
)

score_tail_summary = validation_outcomes.groupby(
    "score_bucket",
    observed=True,
).agg(
    observations=("score", "size"),
    mean_future_percentile=("future_percentile", "mean"),
    mean_future_relative_return=("future_relative_return", "mean"),
    median_future_relative_return=("future_relative_return", "median"),
    positive_relative_return_fraction=(
        "future_relative_return",
        lambda values: values.gt(0).mean(),
    ),
)
score_tail_summary


### Inspect the selected model's feature importance


In [ ]:
importance = pd.Series(
    chosen_model.feature_importances_,
    index=chosen_columns,
    name="importance",
).sort_values(ascending=False)

importance.head(25).sort_values().plot.barh(figsize=(9, 8))
plt.title("Chosen XGBRanker feature importance")
plt.xlabel("Gain importance")
plt.tight_layout()

importance.head(25)


## Save a study result

For a material tuning run, export the executed notebook to HTML under `notebooks/exports/`. Record the chosen trial ID, feature columns, parameters, fold leaderboard, and outer-validation tables in that export. Then restore this notebook to its generic base state.

The locked 2024–2025 test period remains untouched until the feature schema, hyperparameters, candidate rule, and model family are frozen.
